In [1]:
!pip install transformers datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 5.2 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.5.1
    Uninstalling fsspec-2025.5.1:
      Successfully uninstalled fsspec-2025.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.8.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.5.82 which is incompatible.
torch 2.6.0+cu124 re

In [3]:
import pandas as pd
import numpy as np
import torch
from transformers import RobertaTokenizer, RobertaModel
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm import tqdm

# ============================
# 1. Load Dataset
# ============================
df = pd.read_csv("/kaggle/input/roberta/IMDB_Cleaned (1).csv")
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# Split train/valid/test (40k/5k/5k)
X_train, X_temp, y_train, y_temp = train_test_split(
    df['review'], df['sentiment'],
    train_size=40000, stratify=df['sentiment'], random_state=42
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp,
    test_size=5000, stratify=y_temp, random_state=42
)

# ============================
# 2. RoBERTa Embeddings
# ============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
roberta = RobertaModel.from_pretrained("roberta-base").to(device)
roberta.eval()

def get_roberta_embeddings(texts, batch_size=16, max_len=128):
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i+batch_size].tolist()
        encodings = tokenizer(batch_texts, padding=True, truncation=True,
                              max_length=max_len, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = roberta(**encodings)
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(cls_embeddings)
    return np.vstack(all_embeddings)

X_train_roberta = get_roberta_embeddings(X_train)
X_valid_roberta = get_roberta_embeddings(X_valid)
X_test_roberta  = get_roberta_embeddings(X_test)

print("RoBERTa Feature Sizes →", X_train_roberta.shape, X_valid_roberta.shape, X_test_roberta.shape)

# ============================
# 3. Logistic Regression
# ============================
clf = LogisticRegression(max_iter=1000, solver="saga", n_jobs=-1)
clf.fit(X_train_roberta, y_train)

# ============================
# 4. Evaluation
# ============================
def evaluate(model, X, y, split="Data"):
    preds = model.predict(X)
    acc = accuracy_score(y, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(y, preds, average="weighted")
    print(f"{split} → Acc: {acc:.4f}, Prec: {prec:.4f}, Rec: {rec:.4f}, F1: {f1:.4f}")

evaluate(clf, X_train_roberta, y_train, "Train")
evaluate(clf, X_valid_roberta, y_valid, "Valid")
evaluate(clf, X_test_roberta, y_test, "Test")


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 313/313 [00:43<00:00,  7.12it/s]


RoBERTa Feature Sizes → (40000, 768) (5000, 768) (5000, 768)
Train → Acc: 0.8583, Prec: 0.8583, Rec: 0.8583, F1: 0.8583
Valid → Acc: 0.8676, Prec: 0.8676, Rec: 0.8676, F1: 0.8676
Test → Acc: 0.8546, Prec: 0.8547, Rec: 0.8546, F1: 0.8546


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [5]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaModel
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# ============================
# 1. Load Dataset
# ============================
df = pd.read_csv("/kaggle/input/roberta/IMDB_Cleaned (1).csv")
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# Split train/valid/test (40k/5k/5k)
X_train, X_temp, y_train, y_temp = train_test_split(
    df['review'], df['sentiment'],
    train_size=40000, stratify=df['sentiment'], random_state=42
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp,
    test_size=5000, stratify=y_temp, random_state=42
)

# ============================
# 2. TF-IDF Features
# ============================
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train).toarray()
X_valid_tfidf = tfidf.transform(X_valid).toarray()
X_test_tfidf  = tfidf.transform(X_test).toarray()

# ============================
# 3. Dataset Class
# ============================
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

class HybridDataset(Dataset):
    def __init__(self, texts, tfidf_feats, labels, tokenizer, max_len=128):
        # Always convert to list/numpy for safe indexing
        self.texts = texts.tolist() if hasattr(texts, "tolist") else list(texts)
        self.tfidf = tfidf_feats
        self.labels = labels.tolist() if hasattr(labels, "tolist") else list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text, padding="max_length", truncation=True,
            max_length=self.max_len, return_tensors="pt"
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item["tfidf"] = torch.tensor(self.tfidf[idx], dtype=torch.float)
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


train_dataset = HybridDataset(X_train, X_train_tfidf, y_train.values, tokenizer)
valid_dataset = HybridDataset(X_valid, X_valid_tfidf, y_valid.values, tokenizer)
test_dataset  = HybridDataset(X_test,  X_test_tfidf,  y_test.values,  tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=16)
test_loader  = DataLoader(test_dataset, batch_size=16)

# ============================
# 4. Hybrid + RoBERTa Classifier
# ============================
class HybridRobertaClassifier(nn.Module):
    def __init__(self, roberta_model, tfidf_dim, num_classes=2):
        super(HybridRobertaClassifier, self).__init__()
        self.roberta = roberta_model
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(self.roberta.config.hidden_size + tfidf_dim, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, input_ids, attention_mask, tfidf):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_embed = outputs.last_hidden_state[:, 0, :]  # CLS embedding
        combined = torch.cat((cls_embed, tfidf), dim=1)
        x = self.dropout(torch.relu(self.fc1(combined)))
        logits = self.fc2(x)
        return logits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
roberta_base = RobertaModel.from_pretrained("roberta-base")
model = HybridRobertaClassifier(roberta_base, tfidf_dim=5000).to(device)

# ============================
# 5. Training Setup
# ============================
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

def train_epoch(model, loader):
    model.train()
    total_loss, preds, labels = 0, [], []
    for batch in loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        tfidf = batch["tfidf"].to(device)
        targets = batch["labels"].to(device)

        outputs = model(input_ids, attention_mask, tfidf)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
        labels.extend(targets.cpu().numpy())
    acc = accuracy_score(labels, preds)
    return total_loss/len(loader), acc

def eval_epoch(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            tfidf = batch["tfidf"].to(device)
            targets = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask, tfidf)
            preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
            labels.extend(targets.cpu().numpy())
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    return acc, prec, rec, f1

# ============================
# 6. Run Training
# ============================
epochs = 3
for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, train_loader)
    val_acc, val_prec, val_rec, val_f1 = eval_epoch(model, valid_loader)
    print(f"Epoch {epoch+1}/{epochs} → Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, "
          f"Val Acc: {val_acc:.4f}, Prec: {val_prec:.4f}, Rec: {val_rec:.4f}, F1: {val_f1:.4f}")

# ============================
# 7. Final Test Evaluation
# ============================
test_acc, test_prec, test_rec, test_f1 = eval_epoch(model, test_loader)
print(f"\n🏆 Test → Acc: {test_acc:.4f}, Prec: {test_prec:.4f}, Rec: {test_rec:.4f}, F1: {test_f1:.4f}")


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/3 → Train Loss: 0.2920, Train Acc: 0.8785, Val Acc: 0.9028, Prec: 0.9049, Rec: 0.9028, F1: 0.9027
Epoch 2/3 → Train Loss: 0.1955, Train Acc: 0.9234, Val Acc: 0.9144, Prec: 0.9156, Rec: 0.9144, F1: 0.9143
Epoch 3/3 → Train Loss: 0.1309, Train Acc: 0.9525, Val Acc: 0.9070, Prec: 0.9082, Rec: 0.9070, F1: 0.9069

🏆 Test → Acc: 0.9124, Prec: 0.9129, Rec: 0.9124, F1: 0.9124


In [2]:
import pandas as pd
import numpy as np
import torch
from transformers import RobertaTokenizer, RobertaModel
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm import tqdm

# ============================
# 1. Load Dataset
# ============================
df = pd.read_csv("/kaggle/input/roberta/IMDB_Cleaned (1).csv")
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# Split train/valid/test (40k/5k/5k like before)
X_train, X_temp, y_train, y_temp = train_test_split(
    df['review'], df['sentiment'],
    train_size=40000, stratify=df['sentiment'], random_state=42
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp,
    test_size=5000, stratify=y_temp, random_state=42
)

# ============================
# 2. TF-IDF Features
# ============================
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train).toarray()
X_valid_tfidf = tfidf.transform(X_valid).toarray()
X_test_tfidf  = tfidf.transform(X_test).toarray()

# ============================
# 3. RoBERTa Embeddings
# ============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
roberta = RobertaModel.from_pretrained("roberta-base").to(device)
roberta.eval()

def get_roberta_embeddings(texts, batch_size=16, max_len=128):
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i+batch_size].tolist()
        encodings = tokenizer(batch_texts, padding=True, truncation=True,
                              max_length=max_len, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = roberta(**encodings)
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(cls_embeddings)
    return np.vstack(all_embeddings)

X_train_roberta = get_roberta_embeddings(X_train)
X_valid_roberta = get_roberta_embeddings(X_valid)
X_test_roberta  = get_roberta_embeddings(X_test)

# ============================
# 4. Combine Features (TF-IDF + RoBERTa CLS)
# ============================
X_train_hybrid = np.hstack([X_train_tfidf, X_train_roberta])
X_valid_hybrid = np.hstack([X_valid_tfidf, X_valid_roberta])
X_test_hybrid  = np.hstack([X_test_tfidf,  X_test_roberta])

print("Hybrid Feature Sizes →", X_train_hybrid.shape, X_valid_hybrid.shape, X_test_hybrid.shape)

# ============================
# 5. Train Logistic Regression
# ============================
clf = LogisticRegression(max_iter=1000, solver="saga", n_jobs=-1)
clf.fit(X_train_hybrid, y_train)

# ============================
# 6. Evaluate
# ============================
def evaluate(model, X, y, split="Data"):
    preds = model.predict(X)
    acc = accuracy_score(y, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(y, preds, average="weighted")
    print(f"{split} → Acc: {acc:.4f}, Prec: {prec:.4f}, Rec: {rec:.4f}, F1: {f1:.4f}")

evaluate(clf, X_train_hybrid, y_train, "Train")
evaluate(clf, X_valid_hybrid, y_valid, "Valid")
evaluate(clf, X_test_hybrid, y_test, "Test")


2025-09-07 09:20:37.248489: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757236837.501957      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757236837.576401      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 313/313 [00:44<00:00,  7.10it/s]


Hybrid Feature Sizes → (40000, 5768) (5000, 5768) (5000, 5768)
Train → Acc: 0.9215, Prec: 0.9215, Rec: 0.9215, F1: 0.9215
Valid → Acc: 0.9080, Prec: 0.9081, Rec: 0.9080, F1: 0.9080
Test → Acc: 0.9098, Prec: 0.9098, Rec: 0.9098, F1: 0.9098


In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaModel
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# ============================
# 1. Load Dataset
# ============================
# ============================
# 1. Load Dataset
# ============================
df = pd.read_csv("/kaggle/input/roberta/IMDB_Cleaned (1).csv")
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# ============================
# 2. Train/Valid/Test split (fixed sizes)
# ============================
# First split → train (40k) + temp (10k)
X_train, X_temp, y_train, y_temp = train_test_split(
    df['review'], df['sentiment'],
    train_size=40000, stratify=df['sentiment'], random_state=42
)

# Second split → valid (5k) + test (5k)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp,
    test_size=5000, stratify=y_temp, random_state=42
)

# Reset index
X_train, y_train = X_train.reset_index(drop=True), y_train.reset_index(drop=True)
X_valid, y_valid = X_valid.reset_index(drop=True), y_valid.reset_index(drop=True)
X_test,  y_test  = X_test.reset_index(drop=True),  y_test.reset_index(drop=True)

print("✅ Final Data Sizes → Train:", len(X_train), "Valid:", len(X_valid), "Test:", len(X_test))


# ============================
# 2. TF-IDF Features
# ============================
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train).toarray()
X_valid_tfidf = tfidf.transform(X_valid).toarray()
X_test_tfidf  = tfidf.transform(X_test).toarray()

# ============================
# 3. Dataset Class
# ============================
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

class HybridDataset(Dataset):
    def __init__(self, texts, tfidf_feats, labels, tokenizer, max_len=128):
        self.texts = texts
        self.tfidf = tfidf_feats
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text, padding="max_length", truncation=True,
            max_length=self.max_len, return_tensors="pt"
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item["tfidf"] = torch.tensor(self.tfidf[idx], dtype=torch.float)
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = HybridDataset(X_train, X_train_tfidf, y_train.values, tokenizer)
valid_dataset = HybridDataset(X_valid, X_valid_tfidf, y_valid.values, tokenizer)
test_dataset  = HybridDataset(X_test,  X_test_tfidf,  y_test.values,  tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=16)
test_loader  = DataLoader(test_dataset, batch_size=16)

# ============================
# 4. Hybrid Model
# ============================
class HybridRobertaClassifier(nn.Module):
    def __init__(self, roberta_model, tfidf_dim, num_classes=2):
        super(HybridRobertaClassifier, self).__init__()
        self.roberta = roberta_model
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(self.roberta.config.hidden_size + tfidf_dim, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, input_ids, attention_mask, tfidf):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_embed = outputs.last_hidden_state[:, 0, :]  # [CLS] embedding
        combined = torch.cat((cls_embed, tfidf), dim=1)
        x = self.dropout(torch.relu(self.fc1(combined)))
        logits = self.fc2(x)
        return logits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
roberta_base = RobertaModel.from_pretrained("roberta-base")
model = HybridRobertaClassifier(roberta_base, tfidf_dim=5000).to(device)

# ============================
# 5. Training Loop
# ============================
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

def train_epoch(model, loader):
    model.train()
    total_loss, preds, labels = 0, [], []
    for batch in loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        tfidf = batch["tfidf"].to(device)
        targets = batch["labels"].to(device)

        outputs = model(input_ids, attention_mask, tfidf)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
        labels.extend(targets.cpu().numpy())
    acc = accuracy_score(labels, preds)
    return total_loss/len(loader), acc

def eval_epoch(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            tfidf = batch["tfidf"].to(device)
            targets = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask, tfidf)
            preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
            labels.extend(targets.cpu().numpy())
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    return acc, prec, rec, f1

# ============================
# 6. Run Training
# ============================
epochs = 3
for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, train_loader)
    val_acc, val_prec, val_rec, val_f1 = eval_epoch(model, valid_loader)
    print(f"Epoch {epoch+1}/{epochs} → Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, "
          f"Val Acc: {val_acc:.4f}, Prec: {val_prec:.4f}, Rec: {val_rec:.4f}, F1: {val_f1:.4f}")

# ============================
# 7. Final Test Evaluation
# ============================
test_acc, test_prec, test_rec, test_f1 = eval_epoch(model, test_loader)
print(f"\n🏆 Test → Acc: {test_acc:.4f}, Prec: {test_prec:.4f}, Rec: {test_rec:.4f}, F1: {test_f1:.4f}")


✅ Final Data Sizes → Train: 40000 Valid: 5000 Test: 5000


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/3 → Train Loss: 0.2866, Train Acc: 0.8769, Val Acc: 0.9114, Prec: 0.9114, Rec: 0.9114, F1: 0.9114
Epoch 2/3 → Train Loss: 0.1962, Train Acc: 0.9226, Val Acc: 0.9080, Prec: 0.9080, Rec: 0.9080, F1: 0.9080
Epoch 3/3 → Train Loss: 0.1335, Train Acc: 0.9506, Val Acc: 0.9066, Prec: 0.9069, Rec: 0.9066, F1: 0.9066

🏆 Test → Acc: 0.9092, Prec: 0.9098, Rec: 0.9092, F1: 0.9092


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaModel
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# ============================
# 1. Load Dataset
# ============================
df = pd.read_csv("/kaggle/input/roberta/IMDB_Cleaned (1).csv")
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# Train/Val/Test split (fixed sizes)
X_train, X_temp, y_train, y_temp = train_test_split(
    df['review'], df['sentiment'], test_size=0.3, random_state=42, stratify=df['sentiment']
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

# Subset sizes
X_train, y_train = X_train.iloc[:40000].reset_index(drop=True), y_train.iloc[:40000].reset_index(drop=True)
X_valid, y_valid = X_valid.iloc[:5000].reset_index(drop=True), y_valid.iloc[:5000].reset_index(drop=True)
X_test,  y_test  = X_test.iloc[:5000].reset_index(drop=True), y_test.iloc[:5000].reset_index(drop=True)

print("Data Sizes → Train:", len(X_train), "Valid:", len(X_valid), "Test:", len(X_test))

# ============================
# 2. TF-IDF Features
# ============================
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train).toarray()
X_valid_tfidf = tfidf.transform(X_valid).toarray()
X_test_tfidf  = tfidf.transform(X_test).toarray()

# ============================
# 3. Dataset Class
# ============================
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

class HybridDataset(Dataset):
    def __init__(self, texts, tfidf_feats, labels, tokenizer, max_len=128):
        self.texts = texts
        self.tfidf = tfidf_feats
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text, padding="max_length", truncation=True,
            max_length=self.max_len, return_tensors="pt"
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item["tfidf"] = torch.tensor(self.tfidf[idx], dtype=torch.float)
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = HybridDataset(X_train, X_train_tfidf, y_train.values, tokenizer)
valid_dataset = HybridDataset(X_valid, X_valid_tfidf, y_valid.values, tokenizer)
test_dataset  = HybridDataset(X_test,  X_test_tfidf,  y_test.values,  tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=16)
test_loader  = DataLoader(test_dataset, batch_size=16)

# ============================
# 4. Hybrid Model
# ============================
class HybridRobertaClassifier(nn.Module):
    def __init__(self, roberta_model, tfidf_dim, num_classes=2):
        super(HybridRobertaClassifier, self).__init__()
        self.roberta = roberta_model
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(self.roberta.config.hidden_size + tfidf_dim, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, input_ids, attention_mask, tfidf):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_embed = outputs.last_hidden_state[:, 0, :]  # [CLS] embedding
        combined = torch.cat((cls_embed, tfidf), dim=1)
        x = self.dropout(torch.relu(self.fc1(combined)))
        logits = self.fc2(x)
        return logits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
roberta_base = RobertaModel.from_pretrained("roberta-base")
model = HybridRobertaClassifier(roberta_base, tfidf_dim=5000).to(device)

# ============================
# 5. Training Loop
# ============================
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

def train_epoch(model, loader):
    model.train()
    total_loss, preds, labels = 0, [], []
    for batch in loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        tfidf = batch["tfidf"].to(device)
        targets = batch["labels"].to(device)

        outputs = model(input_ids, attention_mask, tfidf)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
        labels.extend(targets.cpu().numpy())
    acc = accuracy_score(labels, preds)
    return total_loss/len(loader), acc

def eval_epoch(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            tfidf = batch["tfidf"].to(device)
            targets = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask, tfidf)
            preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
            labels.extend(targets.cpu().numpy())
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    return acc, prec, rec, f1

# ============================
# 6. Run Training
# ============================
epochs = 3
for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, train_loader)
    val_acc, val_prec, val_rec, val_f1 = eval_epoch(model, valid_loader)
    print(f"Epoch {epoch+1}/{epochs} → Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, "
          f"Val Acc: {val_acc:.4f}, Prec: {val_prec:.4f}, Rec: {val_rec:.4f}, F1: {val_f1:.4f}")

# ============================
# 7. Final Test Evaluation
# ============================
test_acc, test_prec, test_rec, test_f1 = eval_epoch(model, test_loader)
print(f"\n🏆 Test → Acc: {test_acc:.4f}, Prec: {test_prec:.4f}, Rec: {test_rec:.4f}, F1: {test_f1:.4f}")


2025-08-28 09:25:40.471904: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756373140.850345      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756373140.958324      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Data Sizes → Train: 35000 Valid: 5000 Test: 5000


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/3 → Train Loss: 0.2947, Train Acc: 0.8745, Val Acc: 0.9050, Prec: 0.9077, Rec: 0.9050, F1: 0.9049
Epoch 2/3 → Train Loss: 0.2008, Train Acc: 0.9206, Val Acc: 0.9038, Prec: 0.9083, Rec: 0.9038, F1: 0.9035
Epoch 3/3 → Train Loss: 0.1387, Train Acc: 0.9478, Val Acc: 0.9126, Prec: 0.9126, Rec: 0.9126, F1: 0.9126

🏆 Test → Acc: 0.9058, Prec: 0.9058, Rec: 0.9058, F1: 0.9058


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaModel
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# ============================
# 1. Load Dataset
# ============================
df = pd.read_csv("/kaggle/input/roberta/IMDB_Cleaned (1).csv")
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# Train/Val/Test split (fixed sizes)
X_train, X_temp, y_train, y_temp = train_test_split(
    df['review'], df['sentiment'], test_size=0.3, random_state=42, stratify=df['sentiment']
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

# Subset sizes
X_train, y_train = X_train.iloc[:7000].reset_index(drop=True), y_train.iloc[:7000].reset_index(drop=True)
X_valid, y_valid = X_valid.iloc[:2000].reset_index(drop=True), y_valid.iloc[:2000].reset_index(drop=True)
X_test,  y_test  = X_test.iloc[:1000].reset_index(drop=True), y_test.iloc[:1000].reset_index(drop=True)

print("Data Sizes → Train:", len(X_train), "Valid:", len(X_valid), "Test:", len(X_test))

# ============================
# 2. TF-IDF Features
# ============================
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train).toarray()
X_valid_tfidf = tfidf.transform(X_valid).toarray()
X_test_tfidf  = tfidf.transform(X_test).toarray()

# ============================
# 3. Dataset Class
# ============================
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

class HybridDataset(Dataset):
    def __init__(self, texts, tfidf_feats, labels, tokenizer, max_len=128):
        self.texts = texts
        self.tfidf = tfidf_feats
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text, padding="max_length", truncation=True,
            max_length=self.max_len, return_tensors="pt"
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item["tfidf"] = torch.tensor(self.tfidf[idx], dtype=torch.float)
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = HybridDataset(X_train, X_train_tfidf, y_train.values, tokenizer)
valid_dataset = HybridDataset(X_valid, X_valid_tfidf, y_valid.values, tokenizer)
test_dataset  = HybridDataset(X_test,  X_test_tfidf,  y_test.values,  tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=16)
test_loader  = DataLoader(test_dataset, batch_size=16)

# ============================
# 4. Hybrid Model
# ============================
class HybridRobertaClassifier(nn.Module):
    def __init__(self, roberta_model, tfidf_dim, num_classes=2):
        super(HybridRobertaClassifier, self).__init__()
        self.roberta = roberta_model
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(self.roberta.config.hidden_size + tfidf_dim, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, input_ids, attention_mask, tfidf):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_embed = outputs.last_hidden_state[:, 0, :]  # [CLS] embedding
        combined = torch.cat((cls_embed, tfidf), dim=1)
        x = self.dropout(torch.relu(self.fc1(combined)))
        logits = self.fc2(x)
        return logits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
roberta_base = RobertaModel.from_pretrained("roberta-base")
model = HybridRobertaClassifier(roberta_base, tfidf_dim=5000).to(device)

# ============================
# 5. Training Loop
# ============================
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

def train_epoch(model, loader):
    model.train()
    total_loss, preds, labels = 0, [], []
    for batch in loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        tfidf = batch["tfidf"].to(device)
        targets = batch["labels"].to(device)

        outputs = model(input_ids, attention_mask, tfidf)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
        labels.extend(targets.cpu().numpy())
    acc = accuracy_score(labels, preds)
    return total_loss/len(loader), acc

def eval_epoch(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            tfidf = batch["tfidf"].to(device)
            targets = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask, tfidf)
            preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
            labels.extend(targets.cpu().numpy())
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    return acc, prec, rec, f1

# ============================
# 6. Run Training
# ============================
epochs = 3
for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, train_loader)
    val_acc, val_prec, val_rec, val_f1 = eval_epoch(model, valid_loader)
    print(f"Epoch {epoch+1}/{epochs} → Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, "
          f"Val Acc: {val_acc:.4f}, Prec: {val_prec:.4f}, Rec: {val_rec:.4f}, F1: {val_f1:.4f}")

# ============================
# 7. Final Test Evaluation
# ============================
test_acc, test_prec, test_rec, test_f1 = eval_epoch(model, test_loader)
print(f"\n🏆 Test → Acc: {test_acc:.4f}, Prec: {test_prec:.4f}, Rec: {test_rec:.4f}, F1: {test_f1:.4f}")


2025-08-28 07:25:15.368660: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756365915.692188      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756365915.787930      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Data Sizes → Train: 7000 Valid: 2000 Test: 1000


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/3 → Train Loss: 0.3882, Train Acc: 0.8177, Val Acc: 0.8805, Prec: 0.8855, Rec: 0.8805, F1: 0.8800
Epoch 2/3 → Train Loss: 0.2253, Train Acc: 0.9130, Val Acc: 0.8920, Prec: 0.8921, Rec: 0.8920, F1: 0.8920
Epoch 3/3 → Train Loss: 0.1512, Train Acc: 0.9450, Val Acc: 0.8890, Prec: 0.8904, Rec: 0.8890, F1: 0.8889

🏆 Test → Acc: 0.8770, Prec: 0.8779, Rec: 0.8770, F1: 0.8770


In [3]:
import pandas as pd

# Final Evaluation
val_acc, val_prec, val_rec, val_f1 = eval_epoch(model, valid_loader)
test_acc, test_prec, test_rec, test_f1 = eval_epoch(model, test_loader)

results_df = pd.DataFrame([{
    "Set": "Validation",
    "Accuracy": val_acc,
    "Precision": val_prec,
    "Recall": val_rec,
    "F1": val_f1
}, {
    "Set": "Test",
    "Accuracy": test_acc,
    "Precision": test_prec,
    "Recall": test_rec,
    "F1": test_f1
}])

print("\n🏆 Final Results:\n")
print(results_df.round(4))



🏆 Final Results:

          Set  Accuracy  Precision  Recall      F1
0  Validation     0.889     0.8904   0.889  0.8889
1        Test     0.877     0.8779   0.877  0.8770


In [6]:
# ============================
# 1. Import Libraries
# ============================
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ============================
# 2. Load Dataset
# ============================
df = pd.read_csv("/kaggle/input/roberta/IMDB_Cleaned (1).csv")  # change path if needed
print("Dataset shape:", df.shape)
print(df.head())

# ============================
# 3. Train/Valid/Test Split
# ============================
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df["review"], df["sentiment"], train_size=7000, stratify=df["sentiment"], random_state=42
)

valid_texts, test_texts, valid_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=1000, stratify=temp_labels, random_state=42
)

print("Train size:", len(train_texts))
print("Valid size:", len(valid_texts))
print("Test size:", len(test_texts))


Dataset shape: (50000, 4)
                                              review sentiment  \
0  One of the other reviewers has mentioned that ...  positive   
1  A wonderful little production. <br /><br />The...  positive   
2  I thought this was a wonderful way to spend ti...  positive   
3  Basically there's a family where a little boy ...  negative   
4  Petter Mattei's "Love in the Time of Money" is...  positive   

   review_length_words                                     cleaned_review  
0                  307  one of the other reviewers has mentioned that ...  
1                  162  a wonderful little production  the filming tec...  
2                  166  i thought this was a wonderful way to spend ti...  
3                  138  basically there's a family where a little boy ...  
4                  230  petter mattei's  love in the time of money  is...  
Train size: 7000
Valid size: 42000
Test size: 1000
